### Section 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import pyodbc
from datetime import datetime, timezone

### Section 2 — SQL Server connection

In [2]:
from src.database.sql_server import get_sql_connection

conn = get_sql_connection()

print("SQL Server connected successfully!")

SQL Server connected successfully!


In [3]:
cursor = conn.cursor()

cursor.execute("SELECT DB_NAME()")

database_name = cursor.fetchone()[0]

print("Connected database:", database_name)

cursor.close()

Connected database: ClimateImpactDW


### Section 3 — Load Gold data

In [4]:
query = """
SELECT *
FROM gold.fact_environment
"""

df = pd.read_sql(query, conn)

df.head()

C:\Users\admin\AppData\Local\Temp\ipykernel_16632\1774706040.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,environment_key,observation_id,ingestion_id,city_key,date_key,time_key,observation_timestamp_utc,temperature_c,feels_like_c,temp_min_c,...,no,no2,o3,so2,nh3,aqi_category,weather_quality_flag,air_quality_flag,source_system,loaded_at
0,1,WEATHER_d4fabd68-a977-4bc9-9bce-031c249e3263_A...,D4FABD68-A977-4BC9-9BCE-031C249E3263,1,20260823,1320,2026-08-23 13:23:31,29.02,31.88,29.02,...,0.07,2.32,37.71,1.20,0.30,Good,VALID,VALID,OpenWeather,2026-08-23 16:14:45.244
1,2,WEATHER_5b46c956-1865-4363-b7d4-08740a23c78a_A...,5B46C956-1865-4363-B7D4-08740A23C78A,1,20260823,1425,2026-08-23 14:27:10,29.02,31.88,29.02,...,0.00,2.57,35.54,1.33,0.35,Good,VALID,VALID,OpenWeather,2026-08-23 16:14:45.244
2,3,WEATHER_a3cfc3b6-2686-4544-915c-0bd4249c956d_A...,A3CFC3B6-2686-4544-915C-0BD4249C956D,1,20260823,1500,2026-08-23 15:02:02,28.02,30.57,28.02,...,0.00,2.60,33.75,1.45,0.37,Good,VALID,VALID,OpenWeather,2026-08-23 16:14:45.244
3,4,WEATHER_2201c8d1-c2e4-44b7-850a-0e58b5b2dd66_A...,2201C8D1-C2E4-44B7-850A-0E58B5B2DD66,1,20260823,1045,2026-08-23 10:45:52,28.02,30.57,28.02,...,0.12,1.26,43.12,0.98,0.20,Good,VALID,VALID,OpenWeather,2026-08-23 16:14:45.244
4,5,WEATHER_832854a8-60a5-4953-9364-14442012d42a_A...,832854A8-60A5-4953-9364-14442012D42A,1,20260823,1520,2026-08-23 15:24:59,29.02,31.88,29.02,...,0.00,2.60,33.75,1.45,0.37,Good,VALID,VALID,OpenWeather,2026-08-23 16:14:45.244


### Section 4 — Basic profile

In [5]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nData types:")
print(df.dtypes)

Rows: 5799
Columns: 33

Data types:
environment_key                       int64
observation_id                       object
ingestion_id                         object
city_key                              int64
date_key                              int64
time_key                              int64
observation_timestamp_utc    datetime64[ns]
temperature_c                       float64
feels_like_c                        float64
temp_min_c                          float64
temp_max_c                          float64
humidity_pct                        float64
pressure_hpa                        float64
visibility_m                        float64
wind_speed_mps                      float64
wind_direction_deg                  float64
wind_gust_mps                       float64
rainfall_1h_mm                      float64
cloudiness_pct                      float64
openweather_aqi                       int64
pm2_5                               float64
pm10                                floa

### Section 5 — Missing values

In [6]:
missing = (
    df.isna()
      .sum()
      .reset_index()
)

missing.columns = [
    "column",
    "missing_count"
]

missing["missing_pct"] = (
    missing["missing_count"]
    / len(df)
    * 100
)

missing = missing.sort_values(
    "missing_pct",
    ascending=False
)

display(missing)

,column,missing_count,missing_pct
16,wind_gust_mps,4583,79.030867
17,rainfall_1h_mm,4107,70.822556
13,visibility_m,2,0.034489
24,no2,0,0.000000
19,openweather_aqi,0,0.000000
20,pm2_5,0,0.000000
21,pm10,0,0.000000
22,co,0,0.000000
23,no,0,0.000000
25,o3,0,0.000000


### Section 6 — Duplicate observation IDs

In [7]:
duplicate_observations = (
    df["observation_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate observation IDs:",
    duplicate_observations
)

Duplicate observation IDs: 0


### Section 7 — Null observation IDs

In [8]:
null_observation_ids = (
    df["observation_id"]
    .isna()
    .sum()
)

print(
    "Null observation IDs:",
    null_observation_ids
)

Null observation IDs: 0


### Section 8 — Temperature validation

In [9]:
temperature_invalid = df[
    (df["temperature_c"] < -90) |
    (df["temperature_c"] > 60)
]

print(
    "Invalid temperature records:",
    len(temperature_invalid)
)

Invalid temperature records: 0


### Section 9 — Humidity validation

In [10]:
humidity_invalid = df[
    (df["humidity_pct"] < 0) |
    (df["humidity_pct"] > 100)
]

print(
    "Invalid humidity records:",
    len(humidity_invalid)
)

Invalid humidity records: 0


### Section 10 — AQI validation

In [12]:
# OpenWeather's AQI scale is:
# 1 → Good
# 2 → Fair
# 3 → Moderate
# 4 → Poor
# 5 → Very Poor

aqi_invalid = df[
    (~df["openweather_aqi"].isna()) &
    (
        (df["openweather_aqi"] < 1) |
        (df["openweather_aqi"] > 5)
    )
]

print(
    "Invalid AQI records:",
    len(aqi_invalid)
)

Invalid AQI records: 0


### Section 11 — PM2.5 validation

In [13]:
pm25_invalid = df[
    (df["pm2_5"] < 0)
]

print(
    "Negative PM2.5 records:",
    len(pm25_invalid)
)

Negative PM2.5 records: 0


### Section 12 — Wind speed validation

In [14]:
wind_invalid = df[
    (df["wind_speed_mps"] < 0)
]

print(
    "Negative wind-speed records:",
    len(wind_invalid)
)

Negative wind-speed records: 0


### Section 13 — Date range

In [15]:
print(
    "Minimum observation:",
    df["observation_timestamp_utc"].min()
)

print(
    "Maximum observation:",
    df["observation_timestamp_utc"].max()
)

Minimum observation: 2026-08-23 10:13:35
Maximum observation: 2026-08-26 13:37:32


### Section 14 — Cities

In [16]:
city_summary = (
    df.groupby("city_key")
      .agg(
          observations=("observation_id", "count"),
          first_observation=(
              "observation_timestamp_utc",
              "min"
          ),
          last_observation=(
              "observation_timestamp_utc",
              "max"
          )
      )
      .reset_index()
)

display(city_summary)

,city_key,observations,first_observation,last_observation
0,1,290,2026-08-23 10:13:35,2026-08-26 13:29:19
1,2,290,2026-08-23 10:16:12,2026-08-26 13:30:07
2,3,290,2026-08-23 10:22:05,2026-08-26 13:31:52
3,4,290,2026-08-23 10:16:15,2026-08-26 13:32:24
4,5,290,2026-08-23 10:20:55,2026-08-26 13:29:31
5,6,291,2026-08-23 10:17:29,2026-08-26 13:23:48
6,7,290,2026-08-23 10:17:43,2026-08-26 13:30:53
7,8,290,2026-08-23 10:18:25,2026-08-26 13:31:52
8,9,290,2026-08-23 10:18:58,2026-08-26 13:32:04
9,10,290,2026-08-23 10:20:17,2026-08-26 13:29:24


### Section 15 — Final validation summary

In [17]:
validation_summary = {
    "total_rows": len(df),
    "duplicate_observation_ids": duplicate_observations,
    "null_observation_ids": null_observation_ids,
    "invalid_temperature": len(temperature_invalid),
    "invalid_humidity": len(humidity_invalid),
    "invalid_aqi": len(aqi_invalid),
    "negative_pm25": len(pm25_invalid),
    "negative_wind_speed": len(wind_invalid),
    "cities": df["city_key"].nunique()
}

validation_summary

{'total_rows': 5799,
 'duplicate_observation_ids': np.int64(0),
 'null_observation_ids': np.int64(0),
 'invalid_temperature': 0,
 'invalid_humidity': 0,
 'invalid_aqi': 0,
 'negative_pm25': 0,
 'negative_wind_speed': 0,
 'cities': 20}